Test


In [11]:
from google.colab import auth
auth.authenticate_user()
import polars as pl
from datasets import load_dataset
import os
import json

In [10]:

# --- 1. Cấu hình đường dẫn GCS ---
PROJECT_ID = "project-7f16ff1d-9026-4799-bfa"
BUCKET_NAME = "amazon-reviews-lakehouse-data"
listCategory = [
    'Books','CDs_and_Vinyl', 'Cell_Phones_and_Accessories', 'Clothing_Shoes_and_Jewelry',
    'Digital_Music', 'Electronics', 'Gift_Cards', 'Grocery_and_Gourmet_Food',
    'Health_and_Household', 'Home_and_Kitchen', 'Industrial_and_Scientific',
    'Kindle_Store', 'Luxury_Beauty', 'Magazine_Subscriptions', 'Movies_and_TV',
    'Musical_Instruments', 'Office_Products', 'Patio_Lawn_and_Garden',
    'Pet_Supplies', 'Prime_Pantry', 'Software', 'Sports_and_Outdoors',
    'Subscription_Boxes', 'Tools_and_Home_Improvement', 'Toys_and_Games',
    'Video_Games'
]


In [16]:
# def load_review_data(CATEGORY,batch=50000):
#   # Đường dẫn: gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/All_Beauty/
#   GCS_SAVE_PATH = f"gs://{BUCKET_NAME}/bronze-zone/review_data/{CATEGORY}"
#   # --- 2. Load Dataset từ Hugging Face (Streaming) ---
#   dataset = load_dataset(
#       "json",
#       data_files=f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{CATEGORY}.jsonl",
#       split="train",
#       streaming=True
#   )
#   # --- 3. Vòng lặp Batching & Lưu trữ thô ---
#   BATCH_SIZE = batch
#   buffer = []
#   # Danh sách thuộc tính cần giữ lại
#   KEEP_KEYS = ['rating', 'text', 'parent_asin', 'timestamp', 'user_id', 'helpful_vote', 'verified_purchase']

#   print(f"🚀 Đang đẩy dữ liệu thô {CATEGORY} lên GCS (Bronze Zone)...")

#   for i, row in enumerate(dataset):
#       # Chỉ trích xuất các cột mục tiêu, giữ nguyên giá trị gốc
#       buffer.append({k: row.get(k) for k in KEEP_KEYS})

#       if len(buffer) == BATCH_SIZE:
#           # A. Tạo Polars DataFrame từ dữ liệu thô
#           df_batch = pl.DataFrame(buffer)

#           # B. Ghi trực tiếp lên GCS (Không xử lý ETL)
#           batch_num = (i + 1) // BATCH_SIZE
#           file_name = f"{GCS_SAVE_PATH}/batch_{batch_num}.parquet"

#           # Ghi file Parquet với nén mặc định để tiết kiệm dung lượng
#           df_batch.write_parquet(file_name, use_pyarrow=True)

#           print(f"✅ Đã lưu Batch {batch_num} ({i+1} dòng)")
#           buffer = []

#   # Xử lý phần dữ liệu còn sót lại
#   if buffer:
#       final_batch_num = (i // BATCH_SIZE) + 1
#       final_file = f"{GCS_SAVE_PATH}/batch_{final_batch_num}_final.parquet"
#       pl.DataFrame(buffer).write_parquet(final_file, use_pyarrow=True)
#       print(f"✅ Đã lưu Batch cuối cùng.")

#   print(f"🏁 Hoàn thành! Dữ liệu thô đã nằm tại: {GCS_SAVE_PATH}")


In [19]:
def load_review_data(CATEGORY, batch=100000):
    # Đường dẫn lưu trữ cho Review
    GCS_SAVE_PATH = f"gs://{BUCKET_NAME}/bronze-zone/review_data/{CATEGORY}"

    # CHIẾN THUẬT: Đọc dạng "text" để bỏ qua hoàn toàn các trường thừa và lỗi schema
    dataset = load_dataset(
        "text",
        data_files=f"https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/resolve/main/raw/review_categories/{CATEGORY}.jsonl",
        split="train",
        streaming=True
    )

    BATCH_SIZE = batch
    buffer = []
    # Danh sách thuộc tính mục tiêu cho Review
    KEEP_KEYS = ['rating', 'text', 'parent_asin', 'timestamp', 'user_id', 'helpful_vote', 'verified_purchase']

    print(f"🚀 [Text-Parsing Mode] Đang đẩy Review {CATEGORY} lên GCS (Bronze Zone)...")

    try:
        for i, row in enumerate(dataset):
            # 1. Giải mã JSON từ dòng văn bản thô
            try:
                raw_record = json.loads(row['text'])
            except json.JSONDecodeError:
                continue

            # 2. Trích xuất và ép kiểu chuỗi để đảm bảo an toàn cho tầng Bronze
            # Việc ép về string giúp bạn tránh mọi lỗi "Mixed types" khi ghi file Parquet
            record = {}
            for k in KEEP_KEYS:
                val = raw_record.get(k)
                record[k] = str(val) if val is not None else None

            buffer.append(record)

            # 3. Ghi dữ liệu theo Batch
            if len(buffer) == BATCH_SIZE:
                df_batch = pl.DataFrame(buffer)
                batch_num = (i + 1) // BATCH_SIZE
                file_name = f"{GCS_SAVE_PATH}/batch_{batch_num}.parquet"

                df_batch.write_parquet(file_name, use_pyarrow=True)
                print(f"✅ Đã lưu Review Batch {batch_num} ({i+1} dòng)")
                buffer = []

        # Xử lý phần dư cuối cùng
        if buffer:
            final_batch_num = (i // BATCH_SIZE) + 1
            final_file = f"{GCS_SAVE_PATH}/batch_{final_batch_num}_final.parquet"
            pl.DataFrame(buffer).write_parquet(final_file, use_pyarrow=True)
            print(f"✅ Đã lưu Batch cuối cùng cho {CATEGORY}.")

    except Exception as e:
        print(f"❌ Lỗi nghiêm trọng tại danh mục {CATEGORY}: {e}")

    print(f"🏁 Hoàn thành Review: {GCS_SAVE_PATH}")

In [17]:
def read_review_data(CATEGORY,batch=1000):
  print(f'Du lieu {CATEGORY}')
  path = f"gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/{CATEGORY}/*.parquet"
  lazy_df = pl.scan_parquet(path)
  query = lazy_df.head(batch)
  df = query.collect()
  print(df)

In [ ]:
for category in listCategory:
  load_review_data(category,200000)

🚀 [Text-Parsing Mode] Đang đẩy Review Books lên GCS (Bronze Zone)...
✅ Đã lưu Review Batch 1 (200000 dòng)


In [22]:
read_review_data('Books')

Du lieu Books


ComputeError: failed to retrieve first file schema (parquet): expanded paths were empty (path expansion input: 'paths: [Cloud(PlCloudPath { scheme: Gs, uri: "gs://amazon-reviews-lakehouse-data/bronze-zone/review_data/Books/*.parquet" })]', glob: true). Hint: passing a schema can allow this scan to succeed with an empty DataFrame.

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'slice'
	[3] 'sink'
